# 020 — Disaggregation setup & analyses

Sets up and runs the **IML-based** seismic hazard disaggregations for the WP1 sites.

1. Loads the target intensity levels (IMLs) chosen in `017-disagg_imls_for_msa_stripes.ipynb`.
2. Writes one OpenQuake job `.ini` per (IM definition × truncation level × IML).
3. Launches each calculation, **reusing** any whose inputs have not changed.
4. Records every `calc_id` in `wp1/disagg_manifest.json` so the datastores can be
   obtained programmatically later.

## Prerequisites

`003-psha_setup_and_analyses.ipynb` **must have been run first**. This notebook copies
nothing — it reuses what already sits in `hazard_models/eshm20/wp1/`:

| File | Provenance |
|---|---|
| `source_model_logic_tree_eshm20.xml` | untouched ESHM20, placed manually |
| `source_models/` | untouched ESHM20, placed manually |
| `gmpe_logic_tree_AvgSA_0to{3,6}_median_branch.xml` | simplified by hand (single median branch per TRT) |
| `site_model_all_sites.csv` | written from `/results` by notebook 003 |

The target IMLs come from `data_processed/03_site_hazard/AvgSA_{03,06}_imls_for_disaggregation.csv`.
**The AvgSA 0–6 file does not exist yet** — that IM definition is skipped with a warning
until it is created, and picked up automatically once it is.

## Why one config per IML

OpenQuake's `iml_disagg` takes exactly **one** level per IMT, so a separate job is needed
for each target IML. `intensity_measure_types_and_levels` must be absent (the IMTLs are
inferred from `iml_disagg`), and `poes_disagg` cannot be set at the same time.

Each job is **standalone**: it runs its own classical pre-calculation at the single IML
rather than chaining off the PSHA `calc_id`s from 003. The parent PSHA has 25 intensity
levels where these have 1, and that mismatch would break the realization getters.

## Dependencies

**Upstream:** `003-psha_setup_and_analyses.ipynb` (the `wp1/` model),
`017-disagg_imls_for_msa_stripes.ipynb` (the target IMLs).

**Downstream:** the disaggregation post-processing notebook, which reads `disagg-stats`
out of the datastores via `oq_runner.load_calc_ids(...)`. Post-processing is deliberately
**not** done here.

## Runtime

⚠️ At three truncation levels and 11 IMLs this is **66 calculations** (33 until the
AvgSA 0–6 IMLs exist), each a full ERF traversal over the 21.8 MB point-source model at
60 sites, plus the disaggregation itself.

`num_rlzs_disagg = 0` (all 21 SSC realizations) is **required**, not a tuning knob: the
mean disaggregation is only computed when more than one realization is kept. This makes
each run memory-hungry — OpenQuake refuses to start if it cannot fit the accumulator, so
**run one calculation first and check the reported `AccumDict will require X GB`** before
launching the rest.

`DRY_RUN = True` is the default here for that reason.

In [1]:
%load_ext autoreload
%autoreload 2

## 0. Setup & parameters

In [ ]:
import pandas as pd

from phd_project.config import config
from phd_project.scripts import oq_runner

cfg = config.load_config()

In [ ]:
# -----------------------------------------------------------------------------
# PARAMETERS
# -----------------------------------------------------------------------------
WP1_DIR = cfg["hazard_models"]["eshm20_wp1"]
MANIFEST_FP = cfg["hazard_models"]["eshm20_wp1_disagg_manifest"]

# IM period range -> csv of target IMLs. The AvgSA 0-6 file does not exist yet;
# any IM whose file is missing is skipped with a warning.
IML_FILES = {
    "03": cfg["proc_data"]["disagg_imls_AvgSA_03"],
    "06": cfg["proc_data"]["disagg_imls_AvgSA_06"],
}

# Truncation levels (epsilon) to disaggregate at. One calculation is set up and run
# per (IM definition x truncation level x IML).
TRUNCATION_LEVELS = [4]     # truncation levels 3, and 5 have been excluded

# 0 = keep all realizations. Required, not optional: the mean disaggregation
# (disagg-stats) is only computed when more than one realization is kept. Setting
# this to 1 yields the realization *closest to* the mean, not the mean, and
# produces no disagg-stats output at all.
NUM_RLZS_DISAGG = 0

# DRY_RUN:     write configs and report what would run, but launch nothing.
# FORCE_RERUN: re-run every analysis even when its inputs are unchanged.
# NEW_WINDOW:  give each calculation its own console window so its progress can be
#              watched (Windows only; output is teed to WP1_DIR/logs/<name>.log
#              either way). Runs stay sequential regardless.
DRY_RUN = False
FORCE_RERUN = False
NEW_WINDOW = True
# -----------------------------------------------------------------------------

print(f"wp1 dir:  {WP1_DIR}")
print(f"manifest: {MANIFEST_FP}")
print(f"DRY_RUN={DRY_RUN}, FORCE_RERUN={FORCE_RERUN}, NEW_WINDOW={NEW_WINDOW}")

wp1 dir:  C:\Users\clemettn\Documents\phd\hazard_models\eshm20\wp1
manifest: C:\Users\clemettn\Documents\phd\hazard_models\eshm20\wp1\disagg_manifest.json
DRY_RUN=False, FORCE_RERUN=False, NEW_WINDOW=True


## 1. Load the target IMLs

Written by `017-disagg_imls_for_msa_stripes.ipynb`. An IM definition whose file does not
exist yet is skipped rather than treated as an error — this is the expected state for
AvgSA 0–6.

In [4]:
imls_by_im = {}
for im, fp in IML_FILES.items():
    if not fp.is_file():
        print(f"[skip] AvgSA {im}: no IML file at {fp}")
        continue
    imls_by_im[im] = oq_runner.load_imls(fp)
    print(f"AvgSA {im}: {len(imls_by_im[im])} imls from {fp.name}")
    print(f"           {[oq_runner._iml_str(x) for x in imls_by_im[im]]}")

if not imls_by_im:
    raise FileNotFoundError(
        "No IML files found - run 017-disagg_imls_for_msa_stripes.ipynb first.\n"
        + "\n".join(f"  - {fp}" for fp in IML_FILES.values())
    )

n_analyses = sum(len(v) for v in imls_by_im.values()) * len(TRUNCATION_LEVELS)
print(f"\n-> {n_analyses} calculations "
      f"(AvgSA {sorted(imls_by_im)} x eps {TRUNCATION_LEVELS} x imls)")

AvgSA 03: 11 imls from AvgSA_03_imls_for_disaggregation.csv
           ['0.285', '0.31', '0.33', '0.36', '0.4', '0.45', '0.55', '0.65', '0.8', '0.95', '1.1']
[skip] AvgSA 06: no IML file at C:\Users\clemettn\Documents\phd\data_processed\03_site_hazard\AvgSA_06_imls_for_disaggregation.csv

-> 33 calculations (AvgSA ['03'] x eps [3, 4, 5] x imls)


## 2. Write the disaggregation configs

Written flat into `wp1/` alongside the PSHA configs

In [5]:
analyses = {}
for im, imls in imls_by_im.items():
    for eps in TRUNCATION_LEVELS:
        for i, iml in enumerate(imls, start=1):
            name = oq_runner.disagg_analysis_name(im, eps, i)
            analyses[name] = oq_runner.write_disagg_config(
                WP1_DIR / oq_runner.disagg_config_name(im, eps, i),
                description=oq_runner.disagg_description(im, eps, i, iml),
                gsim_logic_tree_file=oq_runner.GMPE_LOGIC_TREES[im],
                truncation_level=eps,
                im_upper=oq_runner.im_upper(im),
                iml=iml,
                num_rlzs_disagg=NUM_RLZS_DISAGG,
            )

print(f"wrote {len(analyses)} configs to {WP1_DIR}\n")
for name in list(analyses)[:3]:
    print(f"  {name:32s} -> {analyses[name].name}")
print(f"  ... ({len(analyses) - 3} more)")

wrote 33 configs to C:\Users\clemettn\Documents\phd\hazard_models\eshm20\wp1

  AvgSA_03_disagg_eps3_iml01       -> config_AvgSA_03_disagg_eps3_iml01.ini
  AvgSA_03_disagg_eps3_iml02       -> config_AvgSA_03_disagg_eps3_iml02.ini
  AvgSA_03_disagg_eps3_iml03       -> config_AvgSA_03_disagg_eps3_iml03.ini
  ... (30 more)


## 3. Run (or reuse) the calculations

Each analysis is launched only if it has not been run before, or if one of its inputs
(config, logic trees, site model, source models) has changed since it was. Runs are
sequential — the engine parallelises internally, so concurrent calculations would only
contend for cores.

⚠️ **Run a single analysis first** and check the engine's reported memory requirement
before letting this loop through all of them. Set `DRY_RUN = False` when ready.

In [6]:
calc_ids = {}
for name, config_fp in analyses.items():
    calc_ids[name] = oq_runner.run_or_reuse(
        name, config_fp, WP1_DIR, MANIFEST_FP,
        force_rerun=FORCE_RERUN, dry_run=DRY_RUN, new_window=NEW_WINDOW,
    )

calc_ids

[oq] 'AvgSA_03_disagg_eps3_iml01' running config_AvgSA_03_disagg_eps3_iml01.ini in a new window (log: logs\AvgSA_03_disagg_eps3_iml01.log) ...
[oq] 'AvgSA_03_disagg_eps3_iml01' completed: calc_id=34
[oq] 'AvgSA_03_disagg_eps3_iml02' running config_AvgSA_03_disagg_eps3_iml02.ini in a new window (log: logs\AvgSA_03_disagg_eps3_iml02.log) ...
[oq] 'AvgSA_03_disagg_eps3_iml02' completed: calc_id=35
[oq] 'AvgSA_03_disagg_eps3_iml03' running config_AvgSA_03_disagg_eps3_iml03.ini in a new window (log: logs\AvgSA_03_disagg_eps3_iml03.log) ...
[oq] 'AvgSA_03_disagg_eps3_iml03' completed: calc_id=36
[oq] 'AvgSA_03_disagg_eps3_iml04' running config_AvgSA_03_disagg_eps3_iml04.ini in a new window (log: logs\AvgSA_03_disagg_eps3_iml04.log) ...
[oq] 'AvgSA_03_disagg_eps3_iml04' completed: calc_id=37
[oq] 'AvgSA_03_disagg_eps3_iml05' running config_AvgSA_03_disagg_eps3_iml05.ini in a new window (log: logs\AvgSA_03_disagg_eps3_iml05.log) ...
[oq] 'AvgSA_03_disagg_eps3_iml05' completed: calc_id=38
[oq] 

{'AvgSA_03_disagg_eps3_iml01': 34,
 'AvgSA_03_disagg_eps3_iml02': 35,
 'AvgSA_03_disagg_eps3_iml03': 36,
 'AvgSA_03_disagg_eps3_iml04': 37,
 'AvgSA_03_disagg_eps3_iml05': 38,
 'AvgSA_03_disagg_eps3_iml06': 39,
 'AvgSA_03_disagg_eps3_iml07': 40,
 'AvgSA_03_disagg_eps3_iml08': 41,
 'AvgSA_03_disagg_eps3_iml09': 42,
 'AvgSA_03_disagg_eps3_iml10': 43,
 'AvgSA_03_disagg_eps3_iml11': 44,
 'AvgSA_03_disagg_eps4_iml01': 45,
 'AvgSA_03_disagg_eps4_iml02': 46,
 'AvgSA_03_disagg_eps4_iml03': 47,
 'AvgSA_03_disagg_eps4_iml04': 48,
 'AvgSA_03_disagg_eps4_iml05': 49,
 'AvgSA_03_disagg_eps4_iml06': 50,
 'AvgSA_03_disagg_eps4_iml07': 51,
 'AvgSA_03_disagg_eps4_iml08': 52,
 'AvgSA_03_disagg_eps4_iml09': 53,
 'AvgSA_03_disagg_eps4_iml10': 54,
 'AvgSA_03_disagg_eps4_iml11': 55,
 'AvgSA_03_disagg_eps5_iml01': 56,
 'AvgSA_03_disagg_eps5_iml02': 57,
 'AvgSA_03_disagg_eps5_iml03': 58,
 'AvgSA_03_disagg_eps5_iml04': 59,
 'AvgSA_03_disagg_eps5_iml05': 60,
 'AvgSA_03_disagg_eps5_iml06': 61,
 'AvgSA_03_disagg_ep

## 4. Manifest

`wp1/disagg_manifest.json` maps each analysis to its `calc_id`, together with the content
hashes of the inputs that produced it. It is small text and **git-tracked** — it is the
pointer that makes the (untracked, machine-local) datastores reproducible.

Downstream notebooks should read it with `oq_runner.load_calc_ids(MANIFEST_FP)` rather
than hardcoding integers.

In [7]:
manifest = oq_runner.load_manifest(MANIFEST_FP)

if not manifest:
    print(f"no manifest yet at {MANIFEST_FP} (nothing has been run)")
else:
    rows = []
    for name, entry in manifest.items():
        # AvgSA_<im>_disagg_eps<n>_iml<nn>
        im, _, eps, iml_i = name.split("_")[1:5]
        rows.append({
            "analysis": name,
            "calc_id": entry["calc_id"],
            "im": im,
            "eps": eps.removeprefix("eps"),
            "iml_i": iml_i.removeprefix("iml"),
            "config": entry["config"],
            "written_at": entry["_meta"]["written_at"],
        })
    display(pd.DataFrame(rows)
            .sort_values(["im", "eps", "iml_i"])
            .reset_index(drop=True))

,analysis,calc_id,im,eps,iml_i,config,written_at
0,AvgSA_03_disagg_eps3_iml01,34,03,3,01,config_AvgSA_03_disagg_eps3_iml01.ini,2026-07-21T08:43:16.233586+00:00
1,AvgSA_03_disagg_eps3_iml02,35,03,3,02,config_AvgSA_03_disagg_eps3_iml02.ini,2026-07-21T08:53:06.632688+00:00
2,AvgSA_03_disagg_eps3_iml03,36,03,3,03,config_AvgSA_03_disagg_eps3_iml03.ini,2026-07-21T09:01:37.601005+00:00
3,AvgSA_03_disagg_eps3_iml04,37,03,3,04,config_AvgSA_03_disagg_eps3_iml04.ini,2026-07-21T09:10:22.839322+00:00
4,AvgSA_03_disagg_eps3_iml05,38,03,3,05,config_AvgSA_03_disagg_eps3_iml05.ini,2026-07-21T09:20:08.078404+00:00
5,AvgSA_03_disagg_eps3_iml06,39,03,3,06,config_AvgSA_03_disagg_eps3_iml06.ini,2026-07-21T09:29:34.974664+00:00
6,AvgSA_03_disagg_eps3_iml07,40,03,3,07,config_AvgSA_03_disagg_eps3_iml07.ini,2026-07-21T09:39:11.246890+00:00
7,AvgSA_03_disagg_eps3_iml08,41,03,3,08,config_AvgSA_03_disagg_eps3_iml08.ini,2026-07-21T09:47:50.733972+00:00
8,AvgSA_03_disagg_eps3_iml09,42,03,3,09,config_AvgSA_03_disagg_eps3_iml09.ini,2026-07-21T09:56:27.961479+00:00
9,AvgSA_03_disagg_eps3_iml10,43,03,3,10,config_AvgSA_03_disagg_eps3_iml10.ini,2026-07-21T10:04:39.230224+00:00
